# Experiment 12 — Voting Ensemble 4-Model (XGBoost + LightGBM + Random Forest + CatBoost)

Combines predictions of the four main tree-based models using soft voting (average of probabilities).

**RAM/Speed Safety Optimization:**
We load the pre-computed test probabilities from each model's run and average them. This avoids duplicating training time.

**Prerequisite:** Run individual experiments (02-10) first.

In [5]:
import os, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# THRESHOLD SCAN RANGE
THRESHOLDS = np.arange(0.1, 0.91, 0.01)
print("Experiment 12 — Voting Ensemble 4-Model (XGB + LGB + RF + CB)")

Experiment 12 — Voting Ensemble 4-Model (XGB + LGB + RF + CB)


In [9]:
def get_ensemble_metrics_and_probs(v, technique):
    """
    Averages test probabilities from XGBoost, LightGBM, Random Forest, and CatBoost.
    """
    xgb_tech_map = {
        'baseline': 'exp1_xgb',
        'class_weights': 'exp2_xgb',
        'smote': 'exp3_xgb',
        'focal_loss': 'exp4_xgb',
        'threshold': 'exp5_xgb'
    }
    
    tech_map = {
        'baseline': 'baseline',
        'class_weights': 'class_weights',
        'smote': 'smote',
        'focal_loss': 'focal_loss',
        'threshold': 'threshold'
    }

    # Load y_test
    y_test = np.load(os.path.join(RESULTS_DIR, f'labels_version_{v}.npy'))

    # Load probabilities
    p_xgb = np.load(os.path.join(RESULTS_DIR, f'probs_{xgb_tech_map[technique]}_version_{v}.npy'))
    p_lgb = np.load(os.path.join(RESULTS_DIR, f'probs_lgb_{tech_map[technique]}_version_{v}.npy'))
    p_rf  = np.load(os.path.join(RESULTS_DIR, f'probs_rf_{tech_map[technique]}_version_{v}.npy'))
    p_cb  = np.load(os.path.join(RESULTS_DIR, f'probs_cb_{tech_map[technique]}_version_{v}.npy'))

    # Soft Voting: Average probabilities
    probs_ensemble = (p_xgb + p_lgb + p_rf + p_cb) / 4.0

    # Load train times
    t_xgb = pd.read_csv(os.path.join(RESULTS_DIR, f'experiment{list(xgb_tech_map.keys()).index(technique)+1}_{"baseline" if technique=="baseline" else "class_weights" if technique=="class_weights" else "smote" if technique=="smote" else "focal_loss" if technique=="focal_loss" else "threshold"}_xgb.csv'))
    t_lgb = pd.read_csv(os.path.join(RESULTS_DIR, 'experiment7_lightgbm.csv'))
    t_rf  = pd.read_csv(os.path.join(RESULTS_DIR, 'experiment8_random_forest.csv'))
    t_cb  = pd.read_csv(os.path.join(RESULTS_DIR, 'experiment9_catboost.csv'))
    
    time_xgb = t_xgb[t_xgb['Version'] == v]['Train_Time_sec'].values[0]
    time_lgb = t_lgb[(t_lgb['Version'] == v) & (t_lgb['Technique'] == tech_names[technique])]['Train_Time_sec'].values[0]
    time_rf  = t_rf[(t_rf['Version'] == v) & (t_rf['Technique'] == tech_names[technique])]['Train_Time_sec'].values[0]
    time_cb  = t_cb[(t_cb['Version'] == v) & (t_cb['Technique'] == tech_names[technique])]['Train_Time_sec'].values[0]
    
    total_train_time = time_xgb + time_lgb + time_rf + time_cb

    # Determine threshold
    if technique == 'threshold' or technique == 'focal_loss':
        f1_scores = [f1_score(y_test, (probs_ensemble>=t).astype(int), pos_label=1, zero_division=0)
                     for t in THRESHOLDS]
        best_threshold = THRESHOLDS[int(np.argmax(f1_scores))]
        preds = (probs_ensemble >= best_threshold).astype(int)
    else:
        best_threshold = 0.5
        preds = (probs_ensemble >= 0.5).astype(int)

    metrics = compute_all_metrics(y_test, probs_ensemble, preds, total_train_time)
    metrics['Best_Threshold'] = round(best_threshold, 2)
    return metrics, probs_ensemble

In [10]:
techniques = ['baseline', 'class_weights', 'smote', 'focal_loss', 'threshold']
tech_names = {'baseline': 'Baseline', 'class_weights': 'Class Weights',
              'smote': 'SMOTE', 'focal_loss': 'Focal Loss', 'threshold': 'Threshold Opt.'}
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp12-Voting4] Version {v}")
    for tech in techniques:
        try:
            metrics, probs = get_ensemble_metrics_and_probs(v, tech)
            metrics['Version']   = v
            metrics['Technique'] = tech_names[tech]
            all_results.append(metrics)
            np.save(os.path.join(RESULTS_DIR, f'probs_voting4_{tech}_version_{v}.npy'), probs)
            print_metrics(metrics, label=f"Voting4-{tech} V{v}")
        except FileNotFoundError as e:
            print(f"  [{tech}] skipped. Missing components: {e}")

print("\n[Exp12-Voting4] Complete.")


[Exp12-Voting4] Version A
[Voting4-baseline VA] AUC=0.8198 | F1=0.7565 | Signal_Sig=177.5182 | Train_Time=1829.99s
[Voting4-class_weights VA] AUC=0.8195 | F1=0.7468 | Signal_Sig=178.0794 | Train_Time=749.86s
[Voting4-smote VA] AUC=0.8187 | F1=0.7486 | Signal_Sig=178.0151 | Train_Time=1242.22s
[Voting4-focal_loss VA] AUC=0.8152 | F1=0.7664 | Signal_Sig=172.7870 | Train_Time=770.42s
[Voting4-threshold VA] AUC=0.8196 | F1=0.7686 | Signal_Sig=177.5165 | Train_Time=783.17s

[Exp12-Voting4] Version B
[Voting4-baseline VB] AUC=0.8151 | F1=0.1446 | Signal_Sig=25.4578 | Train_Time=497.95s
[Voting4-class_weights VB] AUC=0.8131 | F1=0.3658 | Signal_Sig=24.7617 | Train_Time=701.41s
[Voting4-smote VB] AUC=0.7671 | F1=0.3419 | Signal_Sig=24.4611 | Train_Time=858.6s
[Voting4-focal_loss VB] AUC=0.8135 | F1=0.4005 | Signal_Sig=27.5249 | Train_Time=689.84s
[Voting4-threshold VB] AUC=0.8137 | F1=0.4054 | Signal_Sig=25.2709 | Train_Time=660.64s

[Exp12-Voting4] Version C
[Voting4-baseline VC] AUC=0.7940 

In [11]:
cols = ['Version','Technique','AUC','F1','Signal_Significance','Train_Time_sec']
results_df = pd.DataFrame(all_results)
results_df = results_df[[c for c in cols if c in results_df.columns]]
save_path  = os.path.join(RESULTS_DIR, 'experiment12_voting_4model.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      Technique      AUC       F1  Signal_Significance  Train_Time_sec
      A       Baseline 0.819797 0.756462           177.518205         1829.99
      A  Class Weights 0.819510 0.746789           178.079412          749.86
      A          SMOTE 0.818705 0.748613           178.015108         1242.22
      A     Focal Loss 0.815223 0.766378           172.786953          770.42
      A Threshold Opt. 0.819599 0.768645           177.516536          783.17
      B       Baseline 0.815132 0.144583            25.457823          497.95
      B  Class Weights 0.813133 0.365811            24.761735          701.41
      B          SMOTE 0.767111 0.341943            24.461087          858.60
      B     Focal Loss 0.813535 0.400461            27.524919          689.84
      B Threshold Opt. 0.813705 0.405364            25.270876          660.64
      C       Baseline 0.794037 0.006349             3.000000          649.54
      C  Class Weights 0.779935 0.141943             5.491569   